# Модельные эксперименты

Запускать из корня проекта после main.py. Сравнение сделано по OOF LogLoss на фиксированных разбиениях.


In [1]:
from pathlib import Path
import os

if not Path('main.py').exists() and (Path.cwd().parent / 'main.py').exists():
    os.chdir(Path.cwd().parent)


## Другие архитектуры


In [2]:
import json
from pathlib import Path

import mlflow
import numpy as np
import polars as pl
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
from sklearn.model_selection import StratifiedKFold

from main import FEATURES, adjust_pair, row_indices


ROOT = Path.cwd()
OUTPUTS = ROOT / "outputs"


def nbsvm_prediction(x_train, y_train, x_val) -> np.ndarray:
    positive = np.asarray(x_train[y_train == 1].sum(axis=0)).ravel() + 1
    negative = np.asarray(x_train[y_train == 0].sum(axis=0)).ravel() + 1
    ratio = np.log(positive / positive.sum()) - np.log(negative / negative.sum())
    model = LogisticRegression(C=1, max_iter=1000)
    model.fit(x_train.multiply(ratio).tocsr(), y_train)
    return model.predict_proba(x_val.multiply(ratio).tocsr())[:, 1]


def main() -> None:
    (OUTPUTS / "analysis").mkdir(parents=True, exist_ok=True)
    train = pl.read_parquet(OUTPUTS / "v2/processed/train_participants.parquet").sort(["dialog_id", "participant_index"])
    numeric = train.select(FEATURES).to_numpy().astype(float)
    texts = train["text"].to_list()
    labels = train["is_bot"].to_numpy()
    pair_classes = labels.reshape(-1, 2) @ np.array([2, 1])
    oof = {
        name: np.zeros(len(train))
        for name in ("extra_trees", "hist_gradient_boosting", "word_nbsvm", "char_nbsvm")
    }
    cv = StratifiedKFold(4, shuffle=True, random_state=23)
    for fold, (train_dialogues, val_dialogues) in enumerate(
        cv.split(np.arange(len(pair_classes)), pair_classes)
    ):
        tr = row_indices(train_dialogues)
        va = row_indices(val_dialogues)
        models = {
            "extra_trees": ExtraTreesClassifier(
                n_estimators=500, min_samples_leaf=2, max_features=.8,
                random_state=42 + fold, n_jobs=4
            ),
            "hist_gradient_boosting": HistGradientBoostingClassifier(
                max_iter=350, learning_rate=.04, max_leaf_nodes=15,
                l2_regularization=4, random_state=42 + fold
            ),
        }
        for name, model in models.items():
            model.fit(numeric[tr], labels[tr])
            oof[name][va] = model.predict_proba(numeric[va])[:, 1]

        train_texts = [texts[i] for i in tr]
        val_texts = [texts[i] for i in va]
        for name, analyzer, ngrams, limit in [
            ("word_nbsvm", "word", (1, 2), 50000),
            ("char_nbsvm", "char", (2, 5), 120000),
        ]:
            vectorizer = TfidfVectorizer(
                analyzer=analyzer, ngram_range=ngrams, min_df=2,
                max_features=limit, sublinear_tf=True
            )
            x_train = vectorizer.fit_transform(train_texts)
            x_val = vectorizer.transform(val_texts)
            oof[name][va] = nbsvm_prediction(x_train, labels[tr], x_val)
        print(f"fold {fold + 1}/4 done", flush=True)

    scores = {
        name: {
            "raw": round(float(log_loss(labels, values)), 6),
            "pair_adjusted": round(float(log_loss(labels, adjust_pair(values))), 6),
        }
        for name, values in oof.items()
    }
    baseline = json.loads((OUTPUTS / "v2/results.json").read_text(encoding="utf-8"))
    report = {
        "folds": 4,
        "seed": 23,
        "baseline_selected": baseline["selected_model"],
        "baseline_logloss": baseline["oof_logloss"][baseline["selected_model"]],
        "alternatives": scores,
    }
    path = OUTPUTS / "analysis/alternative_models.json"
    path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
    np.savez(OUTPUTS / "analysis/alternative_oof.npz", **oof)

    database = (OUTPUTS / "mlflow.db").resolve()
    mlflow.set_tracking_uri(f"sqlite:///{database.as_posix()}")
    mlflow.set_experiment("bot_detection")
    for name, values in scores.items():
        with mlflow.start_run(run_name=f"alternative_{name}"):
            mlflow.log_params({"folds": 4, "seed": 23, "architecture": name})
            mlflow.log_metric("oof_logloss", values["raw"])
            mlflow.log_metric("oof_logloss_pair_adjusted", values["pair_adjusted"])
            mlflow.log_artifact(str(path))
    print(json.dumps(report, ensure_ascii=False, indent=2))

main()


fold 1/4 done
fold 2/4 done
fold 3/4 done
fold 4/4 done
{
  "folds": 4,
  "seed": 23,
  "baseline_selected": "blend_pair",
  "baseline_logloss": 0.17739102720950867,
  "alternatives": {
    "extra_trees": {
      "raw": 0.267894,
      "pair_adjusted": 0.239866
    },
    "hist_gradient_boosting": {
      "raw": 0.209798,
      "pair_adjusted": 0.20925
    },
    "word_nbsvm": {
      "raw": 0.55927,
      "pair_adjusted": 0.54544
    },
    "char_nbsvm": {
      "raw": 0.573762,
      "pair_adjusted": 0.569309
    }
  }
}


In [3]:
import json
from pathlib import Path
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

report = json.loads(Path('outputs/analysis/alternative_models.json').read_text(encoding='utf-8'))
labels = list(report['alternatives'])
values = [report['alternatives'][name]['pair_adjusted'] for name in labels]
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.barh(labels[::-1], values[::-1], color='#7055F4')
ax.axvline(report['baseline_logloss'], color='#1876AF', linestyle='--', label='Текущая смесь')
ax.set_xlabel('OOF LogLoss')
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig('outputs/analysis/alternative_models_plot.png', dpi=160)
plt.show()


03_models.ipynb:cell4:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


![Другие модели](../outputs/analysis/alternative_models_plot.png)

ExtraTrees, HistGradientBoosting и две NB-SVM ветки уступили текущей смеси на выбранном разбиении.


## Регуляризация и число деревьев


In [4]:
import json
from pathlib import Path

import mlflow
import numpy as np
import polars as pl
from catboost import CatBoostClassifier
from scipy.sparse import hstack
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
from sklearn.model_selection import StratifiedKFold

from main import FEATURES, adjust_pair, new_char_vectorizer, new_word_vectorizer, row_indices


ROOT = Path.cwd()
OUTPUTS = ROOT / "outputs"
CHECKPOINTS = [300, 450, 600, 800]
CAT_CONFIGS = {
    "depth4_l2_4": (4, 4),
    "depth5_l2_4": (5, 4),
    "depth5_l2_12": (5, 12),
    "depth6_l2_4": (6, 4),
}
TEXT_CS = [.3, 1, 3, 10]


def main() -> None:
    (OUTPUTS / "analysis").mkdir(parents=True, exist_ok=True)
    train = pl.read_parquet(OUTPUTS / "v2/processed/train_participants.parquet").sort(["dialog_id", "participant_index"])
    labels = train["is_bot"].to_numpy()
    numeric = train.select(FEATURES).to_numpy().astype(float)
    texts = train["text"].to_list()
    pair_classes = labels.reshape(-1, 2) @ np.array([2, 1])
    cat_oof = {
        f"{name}_iter{iteration}": np.zeros(len(train))
        for name in CAT_CONFIGS for iteration in CHECKPOINTS
    }
    cat_train_losses = {name: [] for name in cat_oof}
    text_oof = {str(c): np.zeros(len(train)) for c in TEXT_CS}
    text_train_losses = {str(c): [] for c in TEXT_CS}
    cv = StratifiedKFold(4, shuffle=True, random_state=23)
    for fold, (train_dialogues, val_dialogues) in enumerate(
        cv.split(np.arange(len(pair_classes)), pair_classes)
    ):
        tr = row_indices(train_dialogues)
        va = row_indices(val_dialogues)
        train_texts = [texts[i] for i in tr]
        val_texts = [texts[i] for i in va]
        char_vectorizer = new_char_vectorizer()
        word_vectorizer = new_word_vectorizer()
        char_train = char_vectorizer.fit_transform(train_texts)
        char_val = char_vectorizer.transform(val_texts)
        word_train = word_vectorizer.fit_transform(train_texts)
        word_val = word_vectorizer.transform(val_texts)
        svd = TruncatedSVD(n_components=48, n_iter=5, random_state=42 + fold)
        svd_train = svd.fit_transform(char_train)
        svd_val = svd.transform(char_val)
        cat_train = np.hstack([numeric[tr], svd_train])
        cat_val = np.hstack([numeric[va], svd_val])
        for config_name, (depth, l2) in CAT_CONFIGS.items():
            model = CatBoostClassifier(
                iterations=max(CHECKPOINTS), depth=depth, learning_rate=.04,
                l2_leaf_reg=l2, loss_function="Logloss", random_seed=42 + fold,
                thread_count=4, verbose=False,
            )
            model.fit(cat_train, labels[tr])
            for iteration in CHECKPOINTS:
                name = f"{config_name}_iter{iteration}"
                cat_oof[name][va] = model.predict_proba(cat_val, ntree_end=iteration)[:, 1]
                train_pred = model.predict_proba(cat_train, ntree_end=iteration)[:, 1]
                cat_train_losses[name].append(float(log_loss(labels[tr], train_pred)))

        both_train = hstack([word_train, char_train]).tocsr()
        both_val = hstack([word_val, char_val]).tocsr()
        for c in TEXT_CS:
            model = LogisticRegression(C=c, max_iter=1000)
            model.fit(both_train, labels[tr])
            text_oof[str(c)][va] = model.predict_proba(both_val)[:, 1]
            text_train_losses[str(c)].append(float(log_loss(labels[tr], model.predict_proba(both_train)[:, 1])))
        print(f"fold {fold + 1}/4 done", flush=True)

    cat_scores = {
        name: {
            "raw_oof": round(float(log_loss(labels, values)), 6),
            "pair_oof": round(float(log_loss(labels, adjust_pair(values))), 6),
            "mean_fold_train": round(float(np.mean(cat_train_losses[name])), 6),
        }
        for name, values in cat_oof.items()
    }
    main_oof = pl.read_parquet(OUTPUTS / "v2/oof_predictions.parquet").sort(["dialog_id", "participant_index"])
    numeric_base = main_oof["cat_extended"].to_numpy()
    svd_base = main_oof["svd_catboost"].to_numpy()
    text_scores = {}
    for c, probabilities in text_oof.items():
        blend = .95 * (.25 * numeric_base + .75 * svd_base) + .05 * probabilities
        text_scores[c] = {
            "raw_oof": round(float(log_loss(labels, probabilities)), 6),
            "mean_fold_train": round(float(np.mean(text_train_losses[c])), 6),
            "blend_pair_oof": round(float(log_loss(labels, adjust_pair(blend))), 6),
        }

    report = {
        "catboost": cat_scores,
        "text_logistic": text_scores,
        "current_settings": {
            "catboost_svd": "depth5_l2_4_iter600",
            "text_C": 10,
        },
    }
    path = OUTPUTS / "analysis/regularization_study.json"
    path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
    database = (OUTPUTS / "mlflow.db").resolve()
    mlflow.set_tracking_uri(f"sqlite:///{database.as_posix()}")
    mlflow.set_experiment("bot_detection")
    for name, item in cat_scores.items():
        with mlflow.start_run(run_name=f"reg_{name}"):
            mlflow.log_params({"study": "regularization", "configuration": name, "seed": 23, "folds": 4})
            mlflow.log_metric("oof_logloss", item["raw_oof"])
            mlflow.log_metric("oof_logloss_pair_adjusted", item["pair_oof"])
            mlflow.log_metric("mean_fold_train_logloss", item["mean_fold_train"])
    for c, item in text_scores.items():
        with mlflow.start_run(run_name=f"reg_text_C{c}"):
            mlflow.log_params({"study": "regularization", "architecture": "word_char_tfidf", "C": c, "seed": 23})
            mlflow.log_metric("oof_logloss", item["raw_oof"])
            mlflow.log_metric("mean_fold_train_logloss", item["mean_fold_train"])
            mlflow.log_metric("blend_pair_oof_logloss", item["blend_pair_oof"])
    print(json.dumps(report, ensure_ascii=False, indent=2))

main()


fold 1/4 done
fold 2/4 done
fold 3/4 done
fold 4/4 done
{
  "catboost": {
    "depth4_l2_4_iter300": {
      "raw_oof": 0.206056,
      "pair_oof": 0.197532,
      "mean_fold_train": 0.132631
    },
    "depth4_l2_4_iter450": {
      "raw_oof": 0.195999,
      "pair_oof": 0.189219,
      "mean_fold_train": 0.095554
    },
    "depth4_l2_4_iter600": {
      "raw_oof": 0.190199,
      "pair_oof": 0.184563,
      "mean_fold_train": 0.07201
    },
    "depth4_l2_4_iter800": {
      "raw_oof": 0.186854,
      "pair_oof": 0.182828,
      "mean_fold_train": 0.051166
    },
    "depth5_l2_4_iter300": {
      "raw_oof": 0.197459,
      "pair_oof": 0.190534,
      "mean_fold_train": 0.106215
    },
    "depth5_l2_4_iter450": {
      "raw_oof": 0.187761,
      "pair_oof": 0.182328,
      "mean_fold_train": 0.069542
    },
    "depth5_l2_4_iter600": {
      "raw_oof": 0.184413,
      "pair_oof": 0.180772,
      "mean_fold_train": 0.048393
    },
    "depth5_l2_4_iter800": {
      "raw_oof": 0.1840

In [5]:
import json
from pathlib import Path

import mlflow
import numpy as np
import polars as pl
from catboost import CatBoostClassifier
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import log_loss
from sklearn.model_selection import StratifiedKFold

from main import FEATURES, adjust_pair, new_char_vectorizer, row_indices


ROOT = Path.cwd()
OUTPUTS = ROOT / "outputs"


def score(labels: np.ndarray, frame: pl.DataFrame, svd_prob: np.ndarray) -> float:
    blend = .95 * (.25 * frame["cat_extended"].to_numpy() + .75 * svd_prob)
    blend += .05 * frame["word_char_tfidf"].to_numpy()
    return float(log_loss(labels, adjust_pair(blend)))


def run_seed(seed: int, train: pl.DataFrame, labels: np.ndarray) -> dict:
    numeric = train.select(FEATURES).to_numpy().astype(float)
    texts = train["text"].to_list()
    pair_classes = labels.reshape(-1, 2) @ np.array([2, 1])
    cv = StratifiedKFold(4, shuffle=True, random_state=seed)
    alternate = np.zeros(len(labels))
    train_losses = []
    for fold, (train_dialogues, val_dialogues) in enumerate(
        cv.split(np.arange(len(pair_classes)), pair_classes)
    ):
        tr = row_indices(train_dialogues)
        va = row_indices(val_dialogues)
        vectorizer = new_char_vectorizer()
        char_train = vectorizer.fit_transform([texts[i] for i in tr])
        char_val = vectorizer.transform([texts[i] for i in va])
        svd = TruncatedSVD(n_components=48, n_iter=5, random_state=42 + fold)
        fit_x = np.hstack([numeric[tr], svd.fit_transform(char_train)])
        val_x = np.hstack([numeric[va], svd.transform(char_val)])
        model = CatBoostClassifier(
            iterations=800, depth=5, learning_rate=.04, l2_leaf_reg=12,
            loss_function="Logloss", random_seed=42 + fold,
            thread_count=4, verbose=False,
        )
        model.fit(fit_x, labels[tr])
        alternate[va] = model.predict_proba(val_x)[:, 1]
        train_losses.append(float(log_loss(labels[tr], model.predict_proba(fit_x)[:, 1])))
        print(f"seed {seed} fold {fold + 1}/4 done", flush=True)

    source = OUTPUTS / "v2/oof_predictions.parquet" if seed == 23 else OUTPUTS / "seed42/oof_predictions.parquet"
    frame = pl.read_parquet(source).sort(["dialog_id", "participant_index"])
    baseline = score(labels, frame, frame["svd_catboost"].to_numpy())
    candidate = score(labels, frame, alternate)
    return {
        "current_blend_oof": round(baseline, 6),
        "stronger_l2_blend_oof": round(candidate, 6),
        "delta": round(candidate - baseline, 6),
        "stronger_l2_svd_oof": round(float(log_loss(labels, adjust_pair(alternate))), 6),
        "stronger_l2_mean_fold_train": round(float(np.mean(train_losses)), 6),
    }


def main() -> None:
    (OUTPUTS / "analysis").mkdir(parents=True, exist_ok=True)
    train = pl.read_parquet(OUTPUTS / "v2/processed/train_participants.parquet").sort(["dialog_id", "participant_index"])
    labels = train["is_bot"].to_numpy()
    results = {str(seed): run_seed(seed, train, labels) for seed in (23, 42)}
    path = OUTPUTS / "analysis/regularization_blend.json"
    path.write_text(json.dumps(results, ensure_ascii=False, indent=2), encoding="utf-8")
    database = (OUTPUTS / "mlflow.db").resolve()
    mlflow.set_tracking_uri(f"sqlite:///{database.as_posix()}")
    mlflow.set_experiment("bot_detection")
    for seed, result in results.items():
        with mlflow.start_run(run_name=f"reg_blend_l2_12_seed{seed}"):
            mlflow.log_params({"study": "regularization_blend", "seed": seed, "depth": 5, "iterations": 800, "l2_leaf_reg": 12})
            mlflow.log_metric("oof_logloss_pair_adjusted", result["stronger_l2_blend_oof"])
            mlflow.log_metric("delta_vs_current", result["delta"])
    print(json.dumps(results, ensure_ascii=False, indent=2))

main()


seed 23 fold 1/4 done
seed 23 fold 2/4 done
seed 23 fold 3/4 done
seed 23 fold 4/4 done
seed 42 fold 1/4 done
seed 42 fold 2/4 done
seed 42 fold 3/4 done
seed 42 fold 4/4 done
{
  "23": {
    "current_blend_oof": 0.177391,
    "stronger_l2_blend_oof": 0.176645,
    "delta": -0.000746,
    "stronger_l2_svd_oof": 0.179645,
    "stronger_l2_mean_fold_train": 0.055101
  },
  "42": {
    "current_blend_oof": 0.174093,
    "stronger_l2_blend_oof": 0.175178,
    "delta": 0.001086,
    "stronger_l2_svd_oof": 0.178732,
    "stronger_l2_mean_fold_train": 0.05153
  }
}


In [6]:
import json
from pathlib import Path

import mlflow
import numpy as np
import polars as pl
from catboost import CatBoostClassifier
from scipy.sparse import hstack
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
from sklearn.model_selection import StratifiedKFold

from main import FEATURES, adjust_pair, new_char_vectorizer, new_word_vectorizer, row_indices


ROOT = Path.cwd()
OUTPUTS = ROOT / "outputs"
SEED = 7


def catboost(fold: int, iterations: int) -> CatBoostClassifier:
    return CatBoostClassifier(
        iterations=iterations, depth=5, learning_rate=.04, l2_leaf_reg=4,
        loss_function="Logloss", random_seed=42 + fold,
        thread_count=4, verbose=False,
    )


def main() -> None:
    (OUTPUTS / "analysis").mkdir(parents=True, exist_ok=True)
    train = pl.read_parquet(OUTPUTS / "v2/processed/train_participants.parquet").sort(["dialog_id", "participant_index"])
    labels = train["is_bot"].to_numpy()
    numeric = train.select(FEATURES).to_numpy().astype(float)
    texts = train["text"].to_list()
    pair_classes = labels.reshape(-1, 2) @ np.array([2, 1])
    predictions = {name: np.zeros(len(labels)) for name in ("cat_extended", "svd_600", "svd_800", "word_char_tfidf")}
    cv = StratifiedKFold(4, shuffle=True, random_state=SEED)
    for fold, (train_dialogues, val_dialogues) in enumerate(
        cv.split(np.arange(len(pair_classes)), pair_classes)
    ):
        tr = row_indices(train_dialogues)
        va = row_indices(val_dialogues)
        train_texts = [texts[i] for i in tr]
        val_texts = [texts[i] for i in va]
        numeric_model = catboost(fold, 650)
        numeric_model.fit(numeric[tr], labels[tr])
        predictions["cat_extended"][va] = numeric_model.predict_proba(numeric[va])[:, 1]

        char_vectorizer = new_char_vectorizer()
        word_vectorizer = new_word_vectorizer()
        char_train = char_vectorizer.fit_transform(train_texts)
        char_val = char_vectorizer.transform(val_texts)
        word_train = word_vectorizer.fit_transform(train_texts)
        word_val = word_vectorizer.transform(val_texts)
        svd = TruncatedSVD(n_components=48, n_iter=5, random_state=42 + fold)
        fit_x = np.hstack([numeric[tr], svd.fit_transform(char_train)])
        val_x = np.hstack([numeric[va], svd.transform(char_val)])
        text_model = catboost(fold, 800)
        text_model.fit(fit_x, labels[tr])
        for trees in (600, 800):
            predictions[f"svd_{trees}"][va] = text_model.predict_proba(val_x, ntree_end=trees)[:, 1]

        logistic = LogisticRegression(C=10, max_iter=1000)
        logistic.fit(hstack([word_train, char_train]).tocsr(), labels[tr])
        predictions["word_char_tfidf"][va] = logistic.predict_proba(
            hstack([word_val, char_val]).tocsr()
        )[:, 1]
        print(f"fold {fold + 1}/4 done", flush=True)

    result = {}
    for trees in (600, 800):
        blend = .95 * (.25 * predictions["cat_extended"] + .75 * predictions[f"svd_{trees}"])
        blend += .05 * predictions["word_char_tfidf"]
        result[str(trees)] = {
            "svd_pair_oof": round(float(log_loss(labels, adjust_pair(predictions[f"svd_{trees}"]))), 6),
            "blend_pair_oof": round(float(log_loss(labels, adjust_pair(blend))), 6),
        }
    result["delta_800_vs_600"] = round(result["800"]["blend_pair_oof"] - result["600"]["blend_pair_oof"], 6)
    (OUTPUTS / "analysis/iterations_seed7.json").write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
    pl.DataFrame({
        "dialog_id": train["dialog_id"],
        "participant_index": train["participant_index"],
        "is_bot": labels,
        **predictions,
    }).write_parquet(OUTPUTS / "analysis/iterations_seed7_oof.parquet")
    mlflow.set_tracking_uri(f"sqlite:///{(OUTPUTS / 'mlflow.db').resolve().as_posix()}")
    mlflow.set_experiment("bot_detection")
    for trees in (600, 800):
        with mlflow.start_run(run_name=f"iterations_seed7_{trees}"):
            mlflow.log_params({"study": "iteration_validation", "seed": SEED, "svd_iterations": trees})
            mlflow.log_metric("blend_pair_oof_logloss", result[str(trees)]["blend_pair_oof"])
    print(json.dumps(result, ensure_ascii=False, indent=2))

main()


fold 1/4 done
fold 2/4 done
fold 3/4 done
fold 4/4 done
{
  "600": {
    "svd_pair_oof": 0.180968,
    "blend_pair_oof": 0.177599
  },
  "800": {
    "svd_pair_oof": 0.181341,
    "blend_pair_oof": 0.175559
  },
  "delta_800_vs_600": -0.00204
}


Усиление L2 помогло на одном разбиении и ухудшило другое. Для финальной смеси увеличение числа деревьев CatBoost с SVD до 800 улучшило результат на трёх проверенных seed.
